# Tutorial 4: Crypto Trading with Fractional Positions

## The Problem: $67,000 per Bitcoin

Integer share environments assume you buy whole units. At $67,000/BTC, that's unrealistic for most traders — you'd need $67K just to open a 1-unit position. Real crypto exchanges let you buy $100 of BTC (0.00149 BTC).

Spectral-Env-Core's `fractional=True` mode enables dollar-based position sizing:
- **Shares are float64** — you can hold 0.00312 BTC
- **`max_trade_size` is notional USD** — action of +1.0 buys $5,000 worth, not 5,000 coins
- **`max_shares` is max position value in USD** — capped by dollar exposure, not unit count

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectral_env_core import SpectralTradingEnv

In [ ]:
# BTC + ETH crypto environment
env = SpectralTradingEnv(
    num_assets=2,
    num_steps=252,
    initial_price=[67_500.0, 3_200.0],   # BTC, ETH
    drift=[0.40, 0.50],                   # crypto has higher drift
    volatility=[0.65, 0.80],              # and much higher vol
    garch_alpha=0.12,
    garch_beta=0.80,
    jump_intensity=4.0,                   # frequent jumps in crypto
    jump_mean=-0.05,                      # crashes are common
    jump_std=0.10,
    starting_cash=50_000,
    max_shares=25_000,                    # max $25K position per asset
    max_trade_size=5_000,                 # max $5K per trade
    transaction_cost_pct=0.001,           # 10 bps (typical CEX fee)
    # Crypto-specific
    fractional=True,                      # ← enables dollar-based sizing
    # BTC-ETH correlation
    correlation=np.array([
        [1.00, 0.75],
        [0.75, 1.00],
    ]),
)

obs, info = env.reset(seed=7)
print(f"Initial state:")
print(f"  Cash:     ${info['cash']:,.2f}")
print(f"  BTC held: {info['shares'][0]:.6f} BTC (${info['shares'][0] * info['prices'][0]:,.2f})")
print(f"  ETH held: {info['shares'][1]:.6f} ETH (${info['shares'][1] * info['prices'][1]:,.2f})")
print(f"  Fractional mode: {info['fractional']}")

In [ ]:
# Execute a buy: action[0] = 0.5 → buy $2,500 of BTC
action = np.array([0.5, 0.0], dtype=np.float32)  # buy BTC, hold ETH
obs, reward, terminated, truncated, info = env.step(action)

print(f"After buying $2,500 of BTC:")
print(f"  Cash:     ${info['cash']:,.2f}")
print(f"  BTC held: {info['shares'][0]:.6f} BTC (${info['shares'][0] * info['prices'][0]:,.2f})")
print(f"  ETH held: {info['shares'][1]:.6f} ETH (${info['shares'][1] * info['prices'][1]:,.2f})")
print(f"\n  → Bought {info['shares'][0]:.6f} BTC at ${info['prices'][0]:,.2f}/BTC")
print(f"  → That's ${info['shares'][0] * info['prices'][0]:,.2f} worth")

In [ ]:
# Visualise a BTC/ETH episode
env.reset(seed=42)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(env.brownian_path[:, 0], color='orange', lw=1.5)
axes[0].set_title('Bitcoin Price Path (σ=65%, λ_jump=4.0)', fontsize=12)
axes[0].set_ylabel('BTC Price ($)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(env.brownian_path[:, 1], color='purple', lw=1.5)
axes[1].set_title('Ethereum Price Path (σ=80%, correlated with BTC at ρ=0.75)', fontsize=12)
axes[1].set_ylabel('ETH Price ($)')
axes[1].set_xlabel('Trading Day')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: both assets move together (ρ=0.75) but ETH is more volatile.")
print("The agent can learn to use BTC as a 'safer' crypto allocation during")
print("high-volatility regimes, shifting to ETH for upside capture in calm periods.")

## How Fractional Mode Changes the Mechanics

| Aspect | Integer Mode (Equities) | Fractional Mode (Crypto) |
|---|---|---|
| `shares` dtype | int32 | float64 |
| `max_trade_size` meaning | Max shares per trade | Max notional USD per trade |
| `max_shares` meaning | Max share count per asset | Max position value (USD) per asset |
| Trade sizing | action × max_trade_size → round to int | action × max_trade_size / price → fractional units |
| Observation normalisation | shares / max_shares | (shares × price) / max_position_value |

The action space and observation space shapes stay identical — only the interpretation changes. This means the same `SpectralExtractor` and PPO architecture work for both equities and crypto.

---

**Next tutorial:** [05 — Parameter Calibration](05_parameter_calibration.ipynb) — fit environment parameters from real ticker data.